# YOLOv8-Large Wafer Defect Detection - GPU Training Pipeline

**Runtime**: Google Colab A100 (80GB) or T4 (16GB)  
**Framework**: Ultralytics YOLOv8 + MLflow  
**Author**: Rajendar Muddasani  

---

## What This Notebook Does

This notebook trains a **YOLOv8-Large** object detection model (43.7M parameters) to identify
10 types of semiconductor wafer defects. The full pipeline:

| Step | Description | Time (A100) |
|------|-------------|-------------|
| 1 | Install dependencies + GPU check | ~1 min |
| 2 | Generate 20K synthetic wafer defect images | ~5 min |
| 3 | Download MVTec AD industrial defect dataset | ~10 min |
| 4 | Visualize dataset samples with bounding boxes | ~30 sec |
| 5 | Analyze class distribution | ~30 sec |
| 6 | Train YOLOv8-Large (100 epochs) | ~40 min (A100) |
| 7 | Train S/M models for comparison | ~15 min |
| 8 | Evaluate on held-out test set | ~2 min |
| 9 | Export ONNX + TensorRT benchmark | ~5 min |
| 10 | Publication-quality result plots | ~1 min |
| 11 | Inference on test images | ~1 min |
| 12 | Save + download all artifacts | ~1 min |

### Defect Classes (10)

| ID | Class | Description | ID | Class | Description |
|----|-------|-------------|-----|-------|-------------|
| 0 | scratch | Surface scratch marks | 5 | bridge | Solder bridge between pads |
| 1 | particle | Foreign particles | 6 | missing_bond | Missing wire bonds |
| 2 | edge_chip | Chipped wafer edges | 7 | crack | Substrate cracks |
| 3 | void | Voids in solder/oxide | 8 | contamination | Chemical contamination |
| 4 | pattern_shift | Lithography misalignment | 9 | delamination | Layer separation |

### GPU Requirements

- **A100 (recommended)**: 80GB VRAM, batch=16, full augmentation, ~40 min for 100 epochs
- **T4 (fallback)**: 16GB VRAM, batch=8, reduced augmentation, ~90 min for 100 epochs

> **Before running**: Go to Runtime > Change runtime type > Select GPU (A100 preferred)

## Step 1: Environment Setup

Install Ultralytics (YOLOv8), MLflow for experiment tracking, and ONNX for model export.
Then verify GPU availability and determine optimal training parameters based on GPU type.

**Why these packages?**
- `ultralytics`: YOLOv8 training, inference, and export (PyTorch-based)
- `mlflow`: Experiment tracking - logs metrics, params, and artifacts per run
- `onnx + onnxruntime-gpu`: Model export for production deployment (2-5x faster inference)
- `scipy`: Used by our synthetic data generator for defect shape generation

In [ ]:
# Install dependencies (quiet mode to reduce output)
!pip install -q ultralytics mlflow pillow numpy scipy
!pip install -q onnx onnxruntime-gpu

import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    cc = torch.cuda.get_device_capability(0)
    bf16_ok = cc[0] >= 8  # Ampere (A100) supports bfloat16

    print(f'GPU:             {gpu_name}')
    print(f'VRAM:            {vram_gb:.1f} GB')
    print(f'Compute Cap:     {cc[0]}.{cc[1]}')
    print(f'bfloat16:        {bf16_ok}')

    # Auto-configure training parameters based on GPU
    if vram_gb >= 40:  # A100
        BATCH_SIZE = 16
        WORKERS = 4
        print('\n>>> A100 detected: batch=16, full augmentation')
    else:  # T4 or smaller
        BATCH_SIZE = 8
        WORKERS = 2
        print('\n>>> T4 detected: batch=8, reduced augmentation')
else:
    raise RuntimeError(
        'No GPU detected! Go to Runtime > Change runtime type > GPU.\n'
        'This notebook requires GPU for training.'
    )

## Step 2: Generate 20K Synthetic Wafer Defect Images

Our data generator creates photorealistic wafer defect images with pixel-accurate
bounding box annotations. Each image:
- 640x640 pixels (YOLOv8 native resolution)
- Wafer background with realistic die grid patterns
- 1-6 defects per image, randomly placed
- 10 defect classes with distinct visual characteristics
- Gaussian noise + brightness variation for realism

The dataset is automatically split: **70% train / 15% val / 15% test**

**Note**: Generation uses multiprocessing (`n_workers=4`) for speed.
On A100, 20K images take ~5 minutes.

In [ ]:
import os
import sys

# Always do a fresh clone to avoid stale cached files from previous Colab runs
!rm -rf yolo-object-detection
!git clone https://github.com/Rajendar-Muddasani-2/yolo-object-detection.git

os.chdir('yolo-object-detection')
sys.path.insert(0, '.')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

from src.data_generator import generate_dataset

# Generate 20,000 synthetic wafer defect images with YOLO-format annotations
stats = generate_dataset(
    output_dir='data/wafer_defects',
    n_images=20000,
    max_defects_per_image=6,
    seed=42,
    train_ratio=0.70,
    val_ratio=0.15,
    n_workers=4,
)

print(f'\nDataset Summary:')
for k, v in stats.items():
    print(f'  {k}: {v}')


## Step 3: Download MVTec AD Industrial Defect Dataset

**MVTec Anomaly Detection (MVTec AD)** is a benchmark dataset of real-world industrial
defect images from the Technical University of Munich. We use 5 texture categories
that are most relevant to semiconductor wafer inspection:

| Category | Defect Types | Relevance |
|----------|-------------|----------|
| carpet | color stains, cuts, holes | Surface contamination analogues |
| grid | bent, broken, glue | Pattern/grid defect analogues |
| leather | color spots, cuts, folds | Substrate damage analogues |
| tile | cracks, glue strips | Crack/delamination analogues |
| wood | holes, scratches, stains | Void/scratch analogues |

The converter (`mvtec_integration.py`) maps MVTec defect types to our 10 YOLO classes
and merges them with the synthetic data for a combined training set.

> **If MVTec download fails** (license required), training will proceed with
> synthetic data only. Results will still be strong since our generator creates
> diverse, realistic defect patterns.

In [ ]:
import urllib.request
import tarfile
from pathlib import Path

mvtec_dir = Path('data/mvtec_ad')

# Download 5 texture categories most relevant to wafer inspection
MVTEC_TEXTURES = ['carpet', 'grid', 'leather', 'tile', 'wood']
MVTEC_BASE_URL = 'https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f282/download/420938166-1629952094'

# MVTec AD requires accepting a license for commercial use.
# For research/portfolio use, the direct link works.
# Manual download: https://www.mvtec.com/company/research/datasets/mvtec-ad

try:
    for texture in MVTEC_TEXTURES:
        texture_dir = mvtec_dir / texture
        if texture_dir.exists():
            print(f'  {texture}: already downloaded, skipping')
            continue

        url = f'{MVTEC_BASE_URL}/{texture}.tar.xz'
        print(f'  Downloading {texture}...')
        tar_path = f'/tmp/{texture}.tar.xz'
        urllib.request.urlretrieve(url, tar_path)
        mvtec_dir.mkdir(parents=True, exist_ok=True)
        with tarfile.open(tar_path) as tar:
            tar.extractall(path=str(mvtec_dir))
        os.remove(tar_path)
        print(f'  {texture}: extracted')

except Exception as e:
    print(f'\nMVTec download issue: {e}')
    print('Proceeding with synthetic data only (still 20K images).')

# Convert MVTec annotations to YOLO format and merge datasets
if mvtec_dir.exists() and any(mvtec_dir.iterdir()):
    from src.mvtec_integration import convert_mvtec_to_yolo, merge_datasets

    # Convert MVTec ground-truth masks to YOLO bounding boxes
    mvtec_stats = convert_mvtec_to_yolo(
        mvtec_dir=str(mvtec_dir),
        output_dir='data/mvtec_yolo',
        img_size=640,
    )
    print(f'\nMVTec converted: {mvtec_stats}')

    # Merge synthetic + MVTec into a single training set
    merge_stats = merge_datasets(
        synthetic_dir='data/wafer_defects',
        mvtec_dir='data/mvtec_yolo',
        merged_dir='data/merged',
    )
    print(f'Merged dataset: {merge_stats}')
    DATA_YAML = 'data/merged/data.yaml'
else:
    print('Using synthetic-only dataset (20K images)')
    DATA_YAML = 'data/wafer_defects/data.yaml'

print(f'\nTraining config: {DATA_YAML}')

## Step 4: Visualize Training Samples

Before training, we inspect sample images to verify:
- Bounding boxes are correctly placed around defects
- Class labels match visual defect types
- Image quality and defect diversity are sufficient

This visual sanity check is critical - incorrect annotations lead to poor model performance.
Each bounding box is color-coded by defect class.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np
from pathlib import Path

CLASSES = [
    'scratch', 'particle', 'edge_chip', 'void', 'pattern_shift',
    'bridge', 'missing_bond', 'crack', 'contamination', 'delamination',
]
COLORS = plt.cm.tab10(np.linspace(0, 1, 10))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
train_imgs = sorted(Path('data/wafer_defects/train/images').glob('*.jpg'))[:8]

for ax, img_path in zip(axes.flat, train_imgs):
    img = Image.open(img_path)
    ax.imshow(img)

    # Overlay ground-truth bounding boxes from YOLO label file
    lbl_path = Path('data/wafer_defects/train/labels') / f'{img_path.stem}.txt'
    if lbl_path.exists():
        for line in lbl_path.read_text().strip().split('\n'):
            parts = line.split()
            cls_id = int(parts[0])
            cx, cy, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            iw, ih = img.size
            x1, y1 = (cx - w / 2) * iw, (cy - h / 2) * ih
            rect = patches.Rectangle(
                (x1, y1), w * iw, h * ih,
                linewidth=2, edgecolor=COLORS[cls_id], facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, CLASSES[cls_id], fontsize=8,
                    color='white', backgroundcolor=COLORS[cls_id])
    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')

plt.suptitle('Training Samples with Ground Truth Bounding Boxes', fontsize=16, fontweight='bold')
plt.tight_layout()

Path('outputs').mkdir(exist_ok=True)
plt.savefig('outputs/sample_training_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/sample_training_images.png')

## Step 5: Class Distribution Analysis

Balanced class distribution is important for object detection. Heavily imbalanced
classes cause the model to ignore rare defect types. We verify:
- All 10 classes are represented
- No single class dominates excessively
- Distribution across train/val/test splits

Our synthetic generator uses weighted sampling to maintain roughly equal class counts.
The MVTec data (if downloaded) adds additional real-world samples to certain classes.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

Path('outputs').mkdir(exist_ok=True)

# Count defect instances per class across all splits
class_counts = {cls: 0 for cls in CLASSES}
split_counts = {}

for split in ['train', 'val', 'test']:
    lbl_dir = Path('data/wafer_defects') / split / 'labels'
    if not lbl_dir.exists():
        continue
    split_n = 0
    for lbl_path in lbl_dir.glob('*.txt'):
        for line in lbl_path.read_text().strip().split('\n'):
            if line.strip():
                cls_id = int(line.split()[0])
                class_counts[CLASSES[cls_id]] += 1
                split_n += 1
    split_counts[split] = split_n

# Bar chart: defect class distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

bars = ax1.bar(class_counts.keys(), class_counts.values(),
               color=[COLORS[i] for i in range(10)])
ax1.set_title('Defect Class Distribution (All Splits)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Defect Class')
ax1.set_ylabel('Instance Count')
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
for bar, count in zip(bars, class_counts.values()):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
             str(count), ha='center', fontsize=10, fontweight='bold')

# Pie chart: split sizes
ax2.pie(split_counts.values(), labels=[f'{k}\n({v:,})' for k, v in split_counts.items()],
        autopct='%1.1f%%', colors=['#3b82f6', '#22c55e', '#ef4444'])
ax2.set_title('Annotations per Split', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

total = sum(class_counts.values())
print(f'Total annotations: {total:,}')
print(f'Per split: {split_counts}')
print(f'\nClass balance ratio (max/min): {max(class_counts.values()) / max(min(class_counts.values()), 1):.1f}x')

## Step 6: Train YOLOv8-Large (Main Training)

This is the core training step. We train **YOLOv8-Large** (43.7M parameters)
with aggressive augmentation:

| Parameter | Value | Why |
|-----------|-------|-----|
| Model | YOLOv8-Large | Best accuracy/speed tradeoff for production |
| Epochs | 100 | Sufficient for convergence with early stopping |
| Image size | 640x640 | Native YOLO resolution, good for wafer images |
| Batch size | 16 (A100) / 8 (T4) | Maximizes GPU memory utilization |
| Patience | 20 | Early stopping if mAP plateaus for 20 epochs |
| Mosaic | 1.0 | Creates 4-image mosaics - key for small defects |
| Mixup | 0.15 | Blends images for regularization |
| Copy-paste | 0.1 | Synthetic augmentation of defect instances |
| Cosine LR | Yes | Smooth learning rate decay |
| Warmup | 3 epochs | Prevents early training instability |

**Expected results on A100** (based on similar studies):
- mAP@50: 0.85-0.95
- mAP@50:95: 0.65-0.80
- Training time: ~40 minutes

MLflow tracks every metric, parameter, and artifact for reproducibility.

In [ ]:
from ultralytics import YOLO
import mlflow
import time

# MLflow experiment tracking
mlflow.set_tracking_uri('file:///content/yolo-object-detection/mlruns')
mlflow.set_experiment('wafer-defect-yolov8')

# Load YOLOv8-Large pretrained on COCO (43.7M params, 165 layers)
# Transfer learning from COCO gives us strong feature extraction out of the box
model = YOLO('yolov8l.pt')

with mlflow.start_run(run_name='yolov8l-wafer-defects-full'):
    mlflow.log_params({
        'model': 'yolov8l',
        'params_M': 43.7,
        'dataset': 'synthetic_20k+mvtec',
        'epochs': 100,
        'imgsz': 640,
        'batch': BATCH_SIZE,
        'optimizer': 'auto',
        'augmentation': 'mosaic+mixup+copypaste',
    })

    t0 = time.time()

    results = model.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=640,
        batch=BATCH_SIZE,
        patience=20,             # Early stopping
        save=True,
        save_period=10,          # Checkpoint every 10 epochs
        device=0,                # GPU 0
        workers=WORKERS,
        project='runs/detect',
        name='yolov8l_wafer',
        exist_ok=True,
        # === Augmentation ===
        mosaic=1.0,              # 4-image mosaic (critical for small defects)
        mixup=0.15,              # Image blending regularization
        copy_paste=0.1,          # Synthetic defect copy-paste
        degrees=15.0,            # Random rotation up to 15 degrees
        translate=0.2,           # Random translation
        scale=0.5,               # Random scaling
        fliplr=0.5,              # Horizontal flip 50%
        flipud=0.5,              # Vertical flip 50% (wafers are rotation-invariant)
        hsv_h=0.015,             # Hue variation
        hsv_s=0.7,               # Saturation variation
        hsv_v=0.4,               # Value/brightness variation
        # === Learning Rate ===
        lr0=0.01,                # Initial learning rate
        lrf=0.01,                # Final LR = lr0 * lrf
        cos_lr=True,             # Cosine annealing
        warmup_epochs=3,         # 3-epoch warmup
    )

    training_time = time.time() - t0
    print(f'\nTraining completed in {training_time / 60:.1f} minutes')

    # Log final metrics to MLflow
    mlflow.log_metric('training_time_min', training_time / 60)
    mlflow.log_metric('map50', results.results_dict.get('metrics/mAP50(B)', 0))
    mlflow.log_metric('map50_95', results.results_dict.get('metrics/mAP50-95(B)', 0))
    mlflow.log_metric('precision', results.results_dict.get('metrics/precision(B)', 0))
    mlflow.log_metric('recall', results.results_dict.get('metrics/recall(B)', 0))

    print(f"\nmAP@50:    {results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
    print(f"mAP@50:95: {results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")
    print(f"Precision: {results.results_dict.get('metrics/precision(B)', 0):.4f}")
    print(f"Recall:    {results.results_dict.get('metrics/recall(B)', 0):.4f}")

## Step 7: Model Size Comparison (S vs M vs L)

To justify choosing YOLOv8-Large, we train YOLOv8-Small (11.2M) and
YOLOv8-Medium (25.9M) on the same data and compare:

| Model | Params | FLOPs | Typical mAP@50 |
|-------|--------|-------|---------------|
| YOLOv8-S | 11.2M | 28.6G | Good for edge/mobile |
| YOLOv8-M | 25.9M | 78.9G | Balanced choice |
| YOLOv8-L | 43.7M | 165.2G | Best accuracy (our pick) |

We train S/M for 50 epochs each (enough for a fair comparison), while
keeping the same augmentation and data pipeline.

**Key question**: Does the 4x parameter increase from S to L give
proportional accuracy gains on wafer defects?

In [ ]:
# Train YOLOv8-Small and Medium for comparison
comparison_results = {}

for model_size, model_name in [('yolov8s.pt', 'YOLOv8-S'), ('yolov8m.pt', 'YOLOv8-M')]:
    print(f'\n{"=" * 50}')
    print(f'Training {model_name} ({model_size})')
    print(f'{"=" * 50}')

    comp_model = YOLO(model_size)

    with mlflow.start_run(run_name=f'{model_name.lower()}-wafer-comparison'):
        t0 = time.time()
        comp_results = comp_model.train(
            data=DATA_YAML,
            epochs=50,               # 50 epochs for comparison (vs 100 for L)
            imgsz=640,
            batch=32 if 'yolov8s' in model_size else 16,  # S fits larger batch
            patience=15,
            device=0,
            workers=WORKERS,
            project='runs/detect',
            name=f'{model_name.lower()}_wafer',
            exist_ok=True,
            mosaic=1.0,
            cos_lr=True,
        )
        elapsed = time.time() - t0

        comparison_results[model_name] = {
            'mAP50': comp_results.results_dict.get('metrics/mAP50(B)', 0),
            'mAP50_95': comp_results.results_dict.get('metrics/mAP50-95(B)', 0),
            'precision': comp_results.results_dict.get('metrics/precision(B)', 0),
            'recall': comp_results.results_dict.get('metrics/recall(B)', 0),
            'training_time_min': elapsed / 60,
        }
        mlflow.log_metrics(comparison_results[model_name])

# Add YOLOv8-L results from the main training
comparison_results['YOLOv8-L'] = {
    'mAP50': results.results_dict.get('metrics/mAP50(B)', 0),
    'mAP50_95': results.results_dict.get('metrics/mAP50-95(B)', 0),
    'precision': results.results_dict.get('metrics/precision(B)', 0),
    'recall': results.results_dict.get('metrics/recall(B)', 0),
    'training_time_min': training_time / 60,
}

print(f'\n{"=" * 60}')
print(f'{"Model":<12} {"mAP@50":>10} {"mAP@50:95":>12} {"Precision":>10} {"Recall":>10} {"Time":>8}')
print(f'{"-" * 60}')
for name, r in comparison_results.items():
    print(f'{name:<12} {r["mAP50"]:>10.4f} {r["mAP50_95"]:>12.4f} '
          f'{r.get("precision", 0):>10.4f} {r.get("recall", 0):>10.4f} '
          f'{r["training_time_min"]:>6.1f}m')
print(f'{"=" * 60}')

## Step 8: Evaluate on Held-Out Test Set

The test set (15% of data, ~3000 images) was never seen during training or
validation. This gives us an unbiased estimate of real-world performance.

**Metrics explained:**
- **mAP@50**: Mean Average Precision at IoU 0.50 (standard detection metric)
- **mAP@50:95**: Average mAP across IoU thresholds 0.50 to 0.95 (stricter)
- **Precision**: Of all detections, what fraction are correct?
- **Recall**: Of all real defects, what fraction did we find?

For semiconductor inspection, **recall is critical** - missing a defect
is far worse than a false positive (which can be manually reviewed).

In [ ]:
from ultralytics import YOLO
import json

# Load the best weights (from early stopping / best val mAP epoch)
best_model = YOLO('runs/detect/yolov8l_wafer/weights/best.pt')

# Run evaluation on the test split (never seen during training)
test_results = best_model.val(
    data=DATA_YAML,
    split='test',
    batch=16,
    device=0,
    plots=True,       # Generates confusion matrix, PR curves, etc.
    save_json=True,    # COCO-format JSON results for analysis
)

print(f'\n{"=" * 50}')
print(f'TEST SET RESULTS (unseen data)')
print(f'{"=" * 50}')
print(f"  mAP@50:     {test_results.results_dict.get('metrics/mAP50(B)', 0):.4f}")
print(f"  mAP@50:95:  {test_results.results_dict.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"  Precision:  {test_results.results_dict.get('metrics/precision(B)', 0):.4f}")
print(f"  Recall:     {test_results.results_dict.get('metrics/recall(B)', 0):.4f}")

# Per-class analysis
print(f'\nPer-class mAP@50:')
if hasattr(test_results, 'maps'):
    for i, cls_name in enumerate(CLASSES):
        if i < len(test_results.maps):
            print(f'  {cls_name:20s}: {test_results.maps[i]:.4f}')

## Step 9: ONNX Export + Speed Benchmark

For production deployment, we export the trained model to ONNX and compare
inference speed across formats:

| Format | Use Case | Typical Speedup |
|--------|----------|----------------|
| PyTorch (.pt) | Development, fine-tuning | 1x (baseline) |
| ONNX (.onnx) | Cross-platform deployment | 1.5-2x |
| TensorRT (.engine) | NVIDIA GPU production | 3-5x |

ONNX (Open Neural Network Exchange) allows deployment on any framework
(PyTorch, TensorFlow, OpenVINO, etc.). TensorRT further optimizes
for NVIDIA GPUs with kernel fusion and FP16/INT8 quantization.

We benchmark each format with 50 inference runs on 640x640 images.

In [ ]:
import time
import numpy as np
import json
from pathlib import Path

Path('models').mkdir(exist_ok=True)

# Export best model to ONNX format (dynamic batch size, simplified graph)
print('Exporting to ONNX...')
best_model.export(format='onnx', dynamic=True, simplify=True, opset=17)
!cp runs/detect/yolov8l_wafer/weights/best.onnx models/best.onnx
!cp runs/detect/yolov8l_wafer/weights/best.pt models/best.pt
print(f'ONNX model size: {Path("models/best.onnx").stat().st_size / 1e6:.1f} MB')

# === Speed Benchmark ===
dummy_img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
N_WARMUP = 10
N_RUNS = 50

results_bench = {}

for fmt, path in [('PyTorch', 'models/best.pt'), ('ONNX', 'models/best.onnx')]:
    m = YOLO(path)
    # Warmup runs (JIT compilation, CUDA context init)
    for _ in range(N_WARMUP):
        m(dummy_img, verbose=False)
    # Timed benchmark runs
    latencies = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        m(dummy_img, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)

    results_bench[fmt] = {
        'mean_ms': round(np.mean(latencies), 1),
        'p50_ms': round(np.median(latencies), 1),
        'p95_ms': round(np.percentile(latencies, 95), 1),
        'fps': round(1000 / np.mean(latencies), 1),
    }
    print(f"{fmt}: {results_bench[fmt]['mean_ms']}ms mean | "
          f"{results_bench[fmt]['fps']} FPS | "
          f"p95={results_bench[fmt]['p95_ms']}ms")

# Try TensorRT export (FP16) - only works on NVIDIA GPUs with tensorrt Package
try:
    print('\nExporting to TensorRT FP16...')
    best_model.export(format='engine', device=0, half=True)
    trt_model = YOLO('runs/detect/yolov8l_wafer/weights/best.engine')
    for _ in range(N_WARMUP):
        trt_model(dummy_img, verbose=False)
    latencies = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        trt_model(dummy_img, verbose=False)
        latencies.append((time.perf_counter() - t0) * 1000)
    results_bench['TensorRT-FP16'] = {
        'mean_ms': round(np.mean(latencies), 1),
        'p50_ms': round(np.median(latencies), 1),
        'p95_ms': round(np.percentile(latencies, 95), 1),
        'fps': round(1000 / np.mean(latencies), 1),
    }
    print(f"TensorRT-FP16: {results_bench['TensorRT-FP16']['mean_ms']}ms mean | "
          f"{results_bench['TensorRT-FP16']['fps']} FPS")
except Exception as e:
    print(f'TensorRT export skipped: {e}')

with open('outputs/speed_benchmark.json', 'w') as f:
    json.dump(results_bench, f, indent=2)
print('\nBenchmark saved: outputs/speed_benchmark.json')

## Step 10: Publication-Quality Result Visualizations

Generate three display-ready figures:

1. **Model Size Comparison**: Bar charts showing how mAP and training time scale
   with model size (S vs M vs L)
2. **Inference Speed Benchmark**: FPS comparison across PyTorch/ONNX/TensorRT
3. **Per-Class Performance Heatmap**: Which defect types are hardest to detect?

These plots use a professional dark theme suitable for technical presentations
and portfolio showcasing.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('dark_background')
plt.rcParams.update({'font.size': 12, 'font.family': 'sans-serif'})

# === Figure 1: Model Size Comparison (3 subplots) ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = list(comparison_results.keys())
map50s = [comparison_results[m]['mAP50'] for m in models]
map50_95s = [comparison_results[m]['mAP50_95'] for m in models]
times_min = [comparison_results[m]['training_time_min'] for m in models]

axes[0].bar(models, map50s, color=['#3b82f6', '#22c55e', '#ef4444'])
axes[0].set_title('mAP@50 by Model Size', fontweight='bold')
axes[0].set_ylabel('mAP@50')
for i, v in enumerate(map50s):
    axes[0].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

axes[1].bar(models, map50_95s, color=['#3b82f6', '#22c55e', '#ef4444'])
axes[1].set_title('mAP@50:95 by Model Size', fontweight='bold')
axes[1].set_ylabel('mAP@50:95')
for i, v in enumerate(map50_95s):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

axes[2].bar(models, times_min, color=['#3b82f6', '#22c55e', '#ef4444'])
axes[2].set_title('Training Time', fontweight='bold')
axes[2].set_ylabel('Minutes')
for i, v in enumerate(times_min):
    axes[2].text(i, v + 0.5, f'{v:.0f}m', ha='center', fontweight='bold')

plt.suptitle('YOLOv8 Model Size Comparison - Wafer Defect Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# === Figure 2: Speed Benchmark ===
fig, ax = plt.subplots(figsize=(10, 5))
formats = list(results_bench.keys())
fps_vals = [results_bench[f]['fps'] for f in formats]
latency_vals = [results_bench[f]['mean_ms'] for f in formats]

bars = ax.bar(formats, fps_vals, color=['#3b82f6', '#22c55e', '#ef4444'][:len(formats)])
ax.set_title('Inference Speed: PyTorch vs ONNX vs TensorRT', fontsize=14, fontweight='bold')
ax.set_ylabel('Frames per Second (FPS)')
for bar, fps, lat in zip(bars, fps_vals, latency_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{fps:.1f} FPS\n({lat:.1f}ms)', ha='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('outputs/speed_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 11: Inference on Test Images

Run the trained model on unseen test images and visualize predictions
with bounding boxes and confidence scores. This shows:

- How accurately the model localizes defects
- Confidence calibration (are high-confidence predictions correct?)
- How well the model handles multiple defect types per image

We use a confidence threshold of 0.25 (industry standard for YOLOv8).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np
from pathlib import Path

COLORS_HEX = ['#ef4444', '#f97316', '#eab308', '#22c55e', '#06b6d4',
               '#3b82f6', '#8b5cf6', '#ec4899', '#f43f5e', '#14b8a6']

test_images = sorted(Path('data/wafer_defects/test/images').glob('*.jpg'))[:8]
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for ax, img_path in zip(axes.flat, test_images):
    # Run inference with best model
    preds = best_model(str(img_path), conf=0.25, verbose=False)
    img = Image.open(img_path)
    ax.imshow(img)

    n_detections = 0
    for r in preds:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            color = COLORS_HEX[cls_id % len(COLORS_HEX)]
            rect = patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1,
                linewidth=2, edgecolor=color, facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(x1, y1 - 4, f'{CLASSES[cls_id]} {conf:.2f}',
                    fontsize=7, color='white', backgroundcolor=color)
            n_detections += 1

    ax.set_title(f'{n_detections} defects detected', fontsize=10)
    ax.axis('off')

plt.suptitle('YOLOv8-Large Predictions on Test Images (conf >= 0.25)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 12: Collect Artifacts + Results Summary

Gather all output artifacts into a single directory for download.
These include:
- Trained model weights (.pt and .onnx)
- Training curves (loss, mAP per epoch)
- Confusion matrix and PR curves
- Speed benchmark results
- Publication-quality visualization plots

**After this cell**: Download `models/best.pt` and `models/best.onnx`,
then copy them to your local `yolo-object-detection/models/` directory
before pushing to GitHub.

In [ ]:
import shutil
from pathlib import Path
import json

# Copy Ultralytics-generated training curves and plots to outputs/
run_dir = Path('runs/detect/yolov8l_wafer')
for plot_file in run_dir.glob('*.png'):
    shutil.copy2(plot_file, f'outputs/{plot_file.name}')

# Save a complete results summary
summary = {
    'model': 'YOLOv8-Large',
    'params_M': 43.7,
    'dataset': '20K synthetic + MVTec AD',
    'test_metrics': {
        'mAP50': test_results.results_dict.get('metrics/mAP50(B)', 0),
        'mAP50_95': test_results.results_dict.get('metrics/mAP50-95(B)', 0),
        'precision': test_results.results_dict.get('metrics/precision(B)', 0),
        'recall': test_results.results_dict.get('metrics/recall(B)', 0),
    },
    'model_comparison': comparison_results,
    'speed_benchmark': results_bench,
    'training_time_min': training_time / 60,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
}

with open('outputs/results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Print final inventory
print('\n=== OUTPUT ARTIFACTS ===')
for f in sorted(Path('outputs').glob('*')):
    size = f.stat().st_size
    unit = 'KB' if size < 1e6 else 'MB'
    val = size / 1024 if size < 1e6 else size / 1e6
    print(f'  {f.name:40s} {val:6.1f} {unit}')

print(f'\n=== MODEL FILES ===')
for f in sorted(Path('models').glob('*')):
    print(f'  {f.name:40s} {f.stat().st_size / 1e6:6.1f} MB')

print(f'\n=== FINAL RESULTS ===')
print(f'  mAP@50:    {summary["test_metrics"]["mAP50"]:.4f}')
print(f'  mAP@50:95: {summary["test_metrics"]["mAP50_95"]:.4f}')
print(f'  Precision: {summary["test_metrics"]["precision"]:.4f}')
print(f'  Recall:    {summary["test_metrics"]["recall"]:.4f}')
print(f'  GPU:       {summary["gpu"]}')
print(f'  Time:      {summary["training_time_min"]:.1f} minutes')
print(f'\nDownload models/best.pt and models/best.onnx for local deployment.')
print('Copy outputs/ to your local repo for README screenshots.')